In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import xgboost as xgb
from sklearn.metrics import f1_score


In [41]:
# Loading the treated data
train_dataset = pd.read_csv('data/train_dataset_treated.csv')

In [42]:
# Defining the function that will use the model to complete the missing values
# Using the model to complete the test dataset

def complete_dataset(model_name, model):

    test_dataset = pd.read_csv('data/test_dataset_treated.csv')

    test_pred = model.predict(test_dataset)
    test_dataset['Transition'] = test_pred
    test_dataset.head()

    # Dropping all columns but the Transition column
    test_dataset.drop(test_dataset.columns.difference(['Transition']), axis=1, inplace=True)

    # Creating a RowId column to store the index, starting from 1
    test_dataset['RowId'] = np.arange(1, test_dataset.shape[0] + 1)

    # Placing the RowId column in the first position
    cols = test_dataset.columns.tolist()
    cols = cols[-1:] + cols[:-1]
    test_dataset = test_dataset[cols]

    # Transforming the Transition column back to its original values
    replace_map = {'Transition': {0: 'CN-CN', 1: 'AD-AD', 2: 'CN-MCI', 3: 'MCI-AD', 4: 'MCI-MCI'}}
    test_dataset.replace(replace_map, inplace=True)
    test_dataset.head()

    # Saving the test dataset to a csv file
    test_dataset.to_csv('test_predictions_' + model_name + '.csv', index=False)

## Random Forest

In [43]:
# Running a Random Forest Classifier
#model_name = 'random_forest'
X = train_dataset.drop('Transition', axis=1)
y = train_dataset['Transition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)

rfc = RandomForestClassifier(n_estimators=100, random_state=123)
rfc.fit(X_train, y_train)
rfc_pred = rfc.predict(X_test)

In [44]:
# Printing f1 score
print(f1_score(y_test, rfc_pred, average='weighted'))

0.48525269713585084


In [45]:
# Printing the confusion matrix
print("Confusion matrix")
print(confusion_matrix(y_test, rfc_pred))

Confusion matrix
[[ 9  0  2  0  8]
 [ 3  9  0  2  1]
 [ 0  0 18  0  0]
 [ 4  4  2  2  4]
 [ 9  2  1  2  8]]


In [46]:
# Printing the classification report
print("Classification report")
print(classification_report(y_test, rfc_pred))

Classification report
              precision    recall  f1-score   support

           0       0.36      0.47      0.41        19
           1       0.60      0.60      0.60        15
           2       0.78      1.00      0.88        18
           3       0.33      0.12      0.18        16
           4       0.38      0.36      0.37        22

    accuracy                           0.51        90
   macro avg       0.49      0.51      0.49        90
weighted avg       0.48      0.51      0.49        90



In [47]:
# Completing the test dataset
complete_dataset('random_forest', rfc)

## XGBoost

In [48]:
#model_name = 'xgboost'
X = train_dataset.drop('Transition', axis=1)
y = train_dataset['Transition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=123)

xgbc = xgb.XGBClassifier(max_depth = 1, objective='req:squarederror', random_state=123, learning_rate=0.2, n_estimators=52)
xgbc.fit(X_train, y_train)
xgbc_pred = xgbc.predict(X_test)

# Printing f1 score
print(f1_score(y_test, xgbc_pred, average='weighted'))

0.4921805713694323


In [49]:
# Printing f1 score
print(f1_score(y_test, xgbc_pred, average='weighted'))

0.4921805713694323


In [50]:
# Printing the confusion matrix
print("Confusion matrix")
print(confusion_matrix(y_test, xgbc_pred))

Confusion matrix
[[ 7  0  4  0  8]
 [ 3  9  0  2  1]
 [ 0  0 18  0  0]
 [ 3  3  2  5  3]
 [ 7  2  2  4  7]]


In [51]:
# Printing the classification report
print("Classification report")
print(classification_report(y_test, xgbc_pred))

Classification report
              precision    recall  f1-score   support

           0       0.35      0.37      0.36        19
           1       0.64      0.60      0.62        15
           2       0.69      1.00      0.82        18
           3       0.45      0.31      0.37        16
           4       0.37      0.32      0.34        22

    accuracy                           0.51        90
   macro avg       0.50      0.52      0.50        90
weighted avg       0.49      0.51      0.49        90



In [52]:
# Completing the test dataset
complete_dataset('xgboost', xgbc)